# Optuna Merged Viewer

`optuna_jh.db`, `optuna_yi.db`, `optuna_yjs.db` 3개의 DB를 한 번에 읽어 study/trial을 병합한다.
어떤 실험(study)들이 있는지 overview를 확인한다.

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

setup 완료


## 1. `병합중/` 스캔 → `optuna_merged.db` 생성 & study summary 로드

`병합중/` 디렉토리의 모든 `optuna_*.db` 파일을 스캔해 하나의 `optuna_merged.db` 로 병합한다.

- 파일명 `optuna_<owner>_*.db` 에서 owner 를 파싱해 각 study 의 `user_attrs["owner"]` 에 주입
- **study_name 이 겹치면 `AssertionError`** (파일 내부 번호가 충돌하지 않도록 관리할 것)
- 이후 모든 셀은 `optuna_merged.db` 하나만 읽는다

In [2]:
import re
import optuna
import pandas as pd
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)

MERGE_DIR = Path("병합중")
MERGED_DB = Path("optuna_merged.db")

# ---------- 1) 병합중/ 스캔 → optuna_merged.db 재생성 ----------
assert MERGE_DIR.exists(), f"{MERGE_DIR.resolve()} 없음"
src_files = sorted(MERGE_DIR.glob("optuna_*.db"))
assert src_files, f"{MERGE_DIR} 안에 optuna_*.db 파일이 없음"

if MERGED_DB.exists():
    MERGED_DB.unlink()
merged_storage = f"sqlite:///{MERGED_DB.as_posix()}"

seen = {}  # study_name -> 첫 등장 파일
for db_path in src_files:
    m = re.match(r"optuna_([^_]+?)(?:_.+)?\.db$", db_path.name)
    owner = m.group(1) if m else "?"
    src_storage = f"sqlite:///{db_path.as_posix()}"
    for s in optuna.get_all_study_summaries(src_storage):
        sname = s.study_name
        assert sname not in seen, (
            f"study_name 중복: '{sname}' ({seen[sname]} vs {db_path.name})"
        )
        seen[sname] = db_path.name
        optuna.copy_study(
            from_study_name=sname,
            from_storage=src_storage,
            to_storage=merged_storage,
            to_study_name=sname,
        )
        # owner 를 merged DB 의 user_attrs 에 주입 (downstream 에서 사용)
        optuna.load_study(study_name=sname, storage=merged_storage).set_user_attr("owner", owner)

print(f"[merge] {len(src_files)} files → {len(seen)} studies → {MERGED_DB}")

# ---------- 2) optuna_merged.db 에서 study summary 로드 ----------
all_summaries = {}  # owner -> (storage, list[summary])
rows = []
for s in optuna.get_all_study_summaries(merged_storage):
    owner = s.user_attrs.get("owner", "?")
    if owner not in all_summaries:
        all_summaries[owner] = (merged_storage, [])
    all_summaries[owner][1].append(s)
    rows.append({
        "owner": owner,
        "study_name": s.study_name,
        "n_trials": s.n_trials,
        "direction": s.direction.name,
        "best_value": s.best_trial.value if s.best_trial else None,
    })

study_info = (
    pd.DataFrame(rows)
    .sort_values(["owner", "study_name"])
    .reset_index(drop=True)
)
print(f"[load] owners={sorted(all_summaries.keys())} | total studies={len(study_info)}")
study_info

[merge] 44 files → 44 studies → optuna_merged.db
[load] owners=['jh', 'js', 'kj', 'se', 'yi', 'yr'] | total studies=44


,owner,study_name,n_trials,direction,best_value
0,jh,22-001-001,50,MINIMIZE,0.005525
1,jh,3-902-001,300,MINIMIZE,0.008576
2,jh,3-930-001,300,MINIMIZE,0.005536
3,jh,30-906-001,330,MINIMIZE,0.001155
4,jh,30-908-001,330,MINIMIZE,0.001168
5,jh,30-909-001,330,MINIMIZE,0.001339
6,jh,30-910-001,330,MINIMIZE,0.006902
7,jh,30-931-001,330,MINIMIZE,0.001141
8,jh,30-932-001,330,MINIMIZE,0.001154
9,js,20-101-001,330,MINIMIZE,0.001387


### 1-1. 제외할 study 드랍 (하드코딩)

분석에서 제외하고 싶은 study를 `(owner, study_name)` 쌍으로 지정한다.
이후 모든 셀(overview, trial 병합, 조합 매칭)은 드랍된 상태의 `study_info` / `all_summaries`를 사용한다.

In [3]:
# 제외할 study 목록 (하드코딩)
DROP_STUDIES = [
    ("yi", "1-002-001"),
    ("yi", "1-002-002"),
    ("yi", "1-003-001"),
]

_before = len(study_info)

# 1) study_info에서 드랍
drop_mask = study_info.set_index(["owner", "study_name"]).index.isin(DROP_STUDIES)
study_info = study_info[~drop_mask].reset_index(drop=True)

# 2) all_summaries에서도 동일하게 드랍 (downstream의 load_study / trials_dataframe이 제외된 상태로 돌도록)
drop_set = set(DROP_STUDIES)
for owner in list(all_summaries.keys()):
    storage, summaries = all_summaries[owner]
    filtered = [s for s in summaries if (owner, s.study_name) not in drop_set]
    all_summaries[owner] = (storage, filtered)

print(f"드랍: {_before - len(study_info)}개 / 남은 study: {len(study_info)}")
study_info

드랍: 0개 / 남은 study: 44


,owner,study_name,n_trials,direction,best_value
0,jh,22-001-001,50,MINIMIZE,0.005525
1,jh,3-902-001,300,MINIMIZE,0.008576
2,jh,3-930-001,300,MINIMIZE,0.005536
3,jh,30-906-001,330,MINIMIZE,0.001155
4,jh,30-908-001,330,MINIMIZE,0.001168
5,jh,30-909-001,330,MINIMIZE,0.001339
6,jh,30-910-001,330,MINIMIZE,0.006902
7,jh,30-931-001,330,MINIMIZE,0.001141
8,jh,30-932-001,330,MINIMIZE,0.001154
9,js,20-101-001,330,MINIMIZE,0.001387


## 2. Overview: owner별 실험 수 & best value

누가 어떤 study들을 돌렸는지 한눈에 본다.

In [4]:
# owner별 요약
owner_summary = (
    study_info.groupby("owner")
    .agg(
        n_studies=("study_name", "count"),
        total_trials=("n_trials", "sum"),
        best_value=("best_value", "min"),
    )
    .reset_index()
)
print("[owner별 요약]")
display(owner_summary)

# 전체 study 목록 (best_value 오름차순)
print("\n[study 목록 — best_value 오름차순]")
study_info.sort_values("best_value", na_position="last").reset_index(drop=True)

[owner별 요약]


,owner,n_studies,total_trials,best_value
0,jh,9,2630,0.001141
1,js,10,3300,0.001161
2,kj,7,2310,0.001150
3,se,2,660,0.001340
4,yi,11,3300,0.001152
5,yr,5,1650,0.001151



[study 목록 — best_value 오름차순]


,owner,study_name,n_trials,direction,best_value
0,jh,30-931-001,330,MINIMIZE,0.001141
1,kj,50-110-001,330,MINIMIZE,0.001150
2,yr,60-904-001,330,MINIMIZE,0.001151
3,yi,10-073-003,300,MINIMIZE,0.001152
4,jh,30-932-001,330,MINIMIZE,0.001154
5,jh,30-906-001,330,MINIMIZE,0.001155
6,yr,60-901-001,330,MINIMIZE,0.001161
7,js,20-101-013,330,MINIMIZE,0.001161
8,jh,30-908-001,330,MINIMIZE,0.001168
9,js,20-101-014,330,MINIMIZE,0.001175


## 3. 모든 trial을 하나의 DataFrame으로 병합

study_name이 owner별로 겹칠 수 있으므로 `owner`, `study_key`(= `owner/study_name`) 컬럼을 추가한다.

In [5]:
frames = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        df = study.trials_dataframe()
        if df.empty:
            continue
        df.insert(0, "owner", owner)
        df.insert(1, "study_name", s.study_name)
        df.insert(2, "study_key", f"{owner}/{s.study_name}")
        frames.append(df)

trials_df = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
print(f"전체 trial shape: {trials_df.shape}")
trials_df.head()

전체 trial shape: (13850, 75)


,owner,study_name,study_key,number,value,datetime_start,datetime_complete,duration,params_clf_colsample_bytree,params_clf_learning_rate,...,user_attrs_n_feat_clean,user_attrs_n_feat_selected,user_attrs_outlier_args,user_attrs_per_model_oof_rmse,user_attrs_saved_at,user_attrs_selected_cols,user_attrs_train_rmse,user_attrs_val_rmse,state,params_clf_filter_threshold
0,jh,22-001-001,jh/22-001-001,0,0.005553,2026-04-22 03:53:11.751219,2026-04-22 03:59:39.963193,0 days 00:06:28.211974,0.952991,0.010341,...,770,775,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",{'lgbm': 0.005570770661652381},2026-04-22 03:59,"[X1, X4, X5, X6, X7, X8, X9, X10, X11, X12, X1...",0.005553,0.005756,COMPLETE,NaN
1,jh,22-001-001,jh/22-001-001,1,0.005543,2026-04-22 03:59:40.159864,2026-04-22 04:05:00.590627,0 days 00:05:20.430763,0.969239,0.025901,...,661,666,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",{'lgbm': 0.005561272448352173},2026-04-22 04:05,"[X1, X4, X5, X6, X7, X8, X9, X10, X11, X12, X1...",0.005543,0.005753,COMPLETE,NaN
2,jh,22-001-001,jh/22-001-001,2,0.005542,2026-04-22 04:05:00.788884,2026-04-22 04:09:48.116558,0 days 00:04:47.327674,0.801029,0.016640,...,565,570,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",{'lgbm': 0.005564991022782121},2026-04-22 04:09,"[X1, X4, X6, X7, X10, X12, X14, X17, X18, X19,...",0.005542,0.005742,COMPLETE,NaN
3,jh,22-001-001,jh/22-001-001,3,0.005537,2026-04-22 04:09:48.321430,2026-04-22 04:14:26.996164,0 days 00:04:38.674734,0.806697,0.020467,...,565,570,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",{'lgbm': 0.005553390844536965},2026-04-22 04:14,"[X1, X4, X6, X7, X10, X12, X14, X17, X18, X19,...",0.005537,0.005749,COMPLETE,NaN
4,jh,22-001-001,jh/22-001-001,4,0.005558,2026-04-22 04:14:27.204707,2026-04-22 04:20:40.262731,0 days 00:06:13.058024,0.709704,0.049365,...,664,669,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",{'lgbm': 0.0055729709396018816},2026-04-22 04:20,"[X1, X4, X5, X6, X7, X8, X9, X10, X11, X12, X1...",0.005558,0.005764,COMPLETE,NaN


## 4. 전체 DB 통합 Top-N

3개 DB의 모든 COMPLETE trial 중 성능이 가장 좋은 trial을 뽑는다.

In [6]:
TOP_N = 20

complete = trials_df[trials_df["state"] == "COMPLETE"].copy()
# direction은 MINIMIZE 가정 (study_info에서 확인 후 다르면 수정)
top_trials = (
    complete.sort_values("value", ascending=True)
    .loc[:, ["owner", "study_name", "number", "value", "datetime_start"]]
    .head(TOP_N)
    .reset_index(drop=True)
)
top_trials

,owner,study_name,number,value,datetime_start
0,jh,30-931-001,233,0.001141,2026-04-17 09:00:29.654079
1,jh,30-931-001,240,0.001141,2026-04-17 09:21:04.211670
2,jh,30-931-001,321,0.001141,2026-04-17 14:07:56.023009
3,jh,30-931-001,165,0.001141,2026-04-17 06:32:28.244909
4,jh,30-931-001,241,0.001141,2026-04-17 09:24:00.405823
5,jh,30-931-001,269,0.001141,2026-04-17 11:09:14.187887
6,jh,30-931-001,251,0.001141,2026-04-17 10:01:17.327187
7,jh,30-931-001,236,0.001141,2026-04-17 09:09:10.656579
8,jh,30-931-001,281,0.001141,2026-04-17 11:58:19.142341
9,jh,30-931-001,232,0.001141,2026-04-17 08:57:58.122486


## 5. 특정 study만 필터링

분석에 쓸 study만 남긴다. `(owner, study_name)` 쌍으로 지정하면 정렬 순서가 바뀌어도 안전하다.

- `jh / 3-101-001`
- `yi / 1-010-001`
- `yjs / 1-100-001`

In [7]:
KEEP_STUDIES = [
    ("jh",  "3-101-001"),
    ("yi",  "1-010-001"),
    ("yjs", "1-100-001"),
]

keep_idx = pd.MultiIndex.from_tuples(KEEP_STUDIES, names=["owner", "study_name"])

study_mask = study_info.set_index(["owner", "study_name"]).index.isin(keep_idx)
study_info_sel = study_info[study_mask].reset_index(drop=True)

trial_mask = trials_df.set_index(["owner", "study_name"]).index.isin(keep_idx)
trials_sel = trials_df[trial_mask].reset_index(drop=True)

print(f"선택된 study: {len(study_info_sel)} / 전체: {len(study_info)}")
print(f"선택된 trial: {len(trials_sel)} / 전체: {len(trials_df)}")
display(study_info_sel)

선택된 study: 0 / 전체: 44
선택된 trial: 0 / 전체: 13850


,owner,study_name,n_trials,direction,best_value


## 6. 조합표(experiment_combinations.xlsx) 매칭

`experiment_combinations.xlsx`의 36개 조합 각각이 **실행됐는지 / 성능이 얼마인지**를 한눈에 본다.

**매칭 키** (study `user_attrs` 기준):

| 조합표 컬럼 | user_attrs 위치 |
|---|---|
| `run_clf` | `pipeline_config.run_clf` |
| `reg_level` | `pipeline_config.reg_level` |
| `TARGET_TRANSFORM` | `target_transform` |
| `CLIP_Y_EXTREME` | `clip_y_extreme` |
| `clf_output` | `pipeline_config.clf_output` (run_clf=False 이면 무시) |

- 한 조합에 매칭되는 study가 여러 개면 **val_rmse가 가장 낮은 것**을 채택
- 날짜는 study `datetime_start`에서 추출
- val_rmse/test_rmse/user는 study user_attrs의 `final_val_rmse`/`final_test_rmse`/`user`

In [8]:
# 6-1. 조합표 로드 + 각 study의 config를 user_attrs에서 추출
combos = pd.read_excel("experiment_combinations.xlsx")
print(f"조합 수: {len(combos)} | 컬럼: {combos.columns.tolist()}")

cfg_rows = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        ua = dict(study.user_attrs)
        pc = ua.get("pipeline_config", {}) or {}
        # final_val_rmse가 기록 안 돼 있으면 best_trial.value로 fallback
        val_rmse = ua.get("final_val_rmse")
        if val_rmse is None and s.best_trial is not None:
            val_rmse = s.best_trial.value
        cfg_rows.append({
            "owner": owner,
            "study_name": s.study_name,
            "exp_id": ua.get("exp_id", s.study_name),
            "run_clf": pc.get("run_clf"),
            "reg_level": pc.get("reg_level"),
            "TARGET_TRANSFORM": ua.get("target_transform"),
            "CLIP_Y_EXTREME": ua.get("clip_y_extreme"),
            "clf_output": pc.get("clf_output") if pc.get("run_clf") else None,
            "clf_filter": pc.get("clf_filter", False),
            "n_trials": s.n_trials,
            "val_rmse": val_rmse,
            "test_rmse": ua.get("final_test_rmse"),
            "user": ua.get("user", owner),
            "날짜": s.datetime_start.date() if s.datetime_start is not None else None,
        })

study_cfg = pd.DataFrame(cfg_rows)
print(f"study config rows: {len(study_cfg)}")
study_cfg

조합 수: 60 | 컬럼: ['run_clf', 'reg_level', 'TARGET_TRANSFORM', 'CLIP_Y_EXTREME', 'clf_output', 'clf_filter', 'exp_id', '날짜', 'val_rmse', 'test_rmse', 'user', 'n_studies']
study config rows: 44


,owner,study_name,exp_id,run_clf,reg_level,TARGET_TRANSFORM,CLIP_Y_EXTREME,clf_output,clf_filter,n_trials,val_rmse,test_rmse,user,날짜
0,jh,22-001-001,22-001-001,True,position,log1p,True,proba,False,50,0.005525,None,jh,2026-04-22
1,jh,3-902-001,3-902-001,True,unit,none,False,binary,True,300,0.008576,None,jh,2026-04-16
2,jh,3-930-001,3-930-001,True,position,none,True,proba,False,300,0.005536,None,jh,2026-04-17
3,jh,30-906-001,30-906-001,True,unit,yeo-johnson,True,proba,False,330,0.001155,None,jh,2026-04-15
4,jh,30-908-001,30-908-001,True,unit,yeo-johnson,False,proba,False,330,0.001168,None,jh,2026-04-16
5,jh,30-909-001,30-909-001,True,unit,yeo-johnson,False,binary,False,330,0.001339,None,jh,2026-04-16
6,jh,30-910-001,30-910-001,True,position,log1p,False,proba,False,330,0.006902,None,jh,2026-04-16
7,jh,30-931-001,30-931-001,False,position,yeo-johnson,True,None,False,330,0.001141,None,jh,2026-04-17
8,jh,30-932-001,30-932-001,True,position,yeo-johnson,True,proba,False,330,0.001154,None,jh,2026-04-17
9,js,20-101-001,20-101-001,True,unit,yeo-johnson,True,binary,False,330,0.001387,None,js,2026-04-16


In [9]:
study_cfg['val_rmse'].sort_values()

7     0.001141
23    0.001150
42    0.001151
35    0.001152
8     0.001154
3     0.001155
39    0.001161
15    0.001161
4     0.001168
16    0.001175
5     0.001339
26    0.001340
13    0.001387
9     0.001387
40    0.005442
30    0.005442
10    0.005453
31    0.005462
24    0.005463
21    0.005466
33    0.005467
34    0.005480
28    0.005516
29    0.005516
0     0.005525
2     0.005536
38    0.005546
37    0.005875
11    0.005880
27    0.005968
32    0.006893
41    0.006893
12    0.006900
6     0.006902
25    0.006910
22    0.006910
14    0.006921
18    0.007253
43    0.008251
36    0.008251
19    0.008257
17    0.008268
20    0.008538
1     0.008576
Name: val_rmse, dtype: float64

In [10]:
# 6-2. 조합표와 study config를 매칭 → 60행 결과표
KEY_COLS = ["run_clf", "reg_level", "TARGET_TRANSFORM", "CLIP_Y_EXTREME", "clf_output", "clf_filter"]

# n_trials가 이 값 이상이면 "완료(O)"로 간주
DONE_TRIAL_THRESHOLD = 100

# clf_output의 NaN(= run_clf=False 일 때)은 merge가 안되므로 sentinel로 치환
NA_SENTINEL = "__NA__"
combos_m = combos[KEY_COLS].copy()
combos_m["clf_output"] = combos_m["clf_output"].fillna(NA_SENTINEL)

study_cfg_m = study_cfg.copy()
study_cfg_m["clf_output"] = study_cfg_m["clf_output"].fillna(NA_SENTINEL)

# 조합 단위로 best(val_rmse 최소) 선택
#   - val_rmse가 fallback(best_trial.value)으로도 채워지므로 dropna 불필요
#   - 그래도 전부 NaN인 study는 밀려나도록 na_position=last
best_per_combo = (
    study_cfg_m.sort_values("val_rmse", ascending=True, na_position="last")
    .drop_duplicates(subset=KEY_COLS, keep="first")
)

# 한 조합에 붙은 study 수 (val_rmse 유무 무관)
match_count = (
    study_cfg_m.groupby(KEY_COLS, dropna=False)
    .size()
    .rename("n_studies")
    .reset_index()
)

result = (
    combos_m.merge(
        best_per_combo[KEY_COLS + ["exp_id", "날짜", "n_trials", "val_rmse", "test_rmse", "user"]],
        on=KEY_COLS,
        how="left",
    )
    .merge(match_count, on=KEY_COLS, how="left")
)
result["n_studies"] = result["n_studies"].fillna(0).astype(int)
result["clf_output"] = result["clf_output"].replace(NA_SENTINEL, None)

# done 상태 3단계
#   O : n_trials >= DONE_TRIAL_THRESHOLD
#   △ : study는 있으나 trial 수가 threshold 미만
#   X : 시도 없음
def _status(row):
    if pd.notna(row["n_trials"]) and row["n_trials"] >= DONE_TRIAL_THRESHOLD:
        return "O"
    if row["n_studies"] > 0:
        return "△"
    return "X"

result["done"] = result.apply(_status, axis=1)

display_cols = KEY_COLS + ["done", "n_studies", "n_trials", "exp_id", "날짜", "val_rmse", "test_rmse", "user"]

n_done = (result["done"] == "O").sum()
n_partial = (result["done"] == "△").sum()
n_todo = (result["done"] == "X").sum()
print(f"완료(O, n_trials≥{DONE_TRIAL_THRESHOLD}): {n_done} | 진행중(△): {n_partial} | 미시도(X): {n_todo} | 총 {len(result)} 조합")

# O → △ → X 순, 동일 상태 내에서는 val_rmse 오름차순
status_order = {"O": 0, "△": 1, "X": 2}
result["_ord"] = result["done"].map(status_order)
result_sorted = (
    result.sort_values(["_ord", "val_rmse"], ascending=[True, True], na_position="last")
    .drop(columns="_ord")
    .reset_index(drop=True)
)
result_sorted[display_cols].head(30)

완료(O, n_trials≥100): 41 | 진행중(△): 0 | 미시도(X): 19 | 총 60 조합


,run_clf,reg_level,TARGET_TRANSFORM,CLIP_Y_EXTREME,clf_output,clf_filter,done,n_studies,n_trials,exp_id,날짜,val_rmse,test_rmse,user
0,False,position,yeo-johnson,True,None,False,O,1,330.0,30-931-001,2026-04-17,0.001141,None,jh
1,False,unit,yeo-johnson,True,None,False,O,1,330.0,50-110-001,2026-04-16,0.001150,None,kj
2,False,position,yeo-johnson,False,None,True,O,1,330.0,60-904-001,2026-04-15,0.001151,None,yr
3,False,position,yeo-johnson,False,None,False,O,1,300.0,10-073-003,2026-04-17,0.001152,None,yi
4,True,position,yeo-johnson,True,proba,False,O,1,330.0,30-932-001,2026-04-17,0.001154,None,jh
5,True,unit,yeo-johnson,True,proba,False,O,1,330.0,30-906-001,2026-04-15,0.001155,None,jh
6,False,unit,yeo-johnson,False,None,True,O,1,330.0,60-901-001,2026-04-15,0.001161,None,yr
7,True,position,yeo-johnson,True,proba,True,O,1,330.0,20-101-013,2026-04-17,0.001161,None,js
8,True,unit,yeo-johnson,False,proba,False,O,1,330.0,30-908-001,2026-04-16,0.001168,None,jh
9,True,position,yeo-johnson,False,proba,True,O,1,330.0,20-101-014,2026-04-17,0.001175,None,js


In [11]:
result_sorted.to_excel("experiment_status_summary.xlsx", index=False)